# Mlektic — Phase 0 review

This notebook validates the mathematical-integrity contract introduced in phase 0. It is intentionally smaller than the exploratory notebooks and focuses on observable, testable behavior.

## Review goals

- distinguish reconstructed replay from synthetic interpolation;
- verify K captured checkpoints versus N displayed checkpoints;
- preserve source coordinates after temporal decimation;
- separate raw loss from display smoothing;
- verify prediction explanations against the fitted estimator;
- identify extrapolation and explicit counterfactual values;
- support string class labels and multiclass figures;
- inspect the HTML export dependency contract.

> The classic style, fixed dimensions, and current animation behavior remain the defaults. Phase 0 adds semantic transparency; it is not the typography or responsive-layout phase.

In [ ]:
from importlib.metadata import version
from pathlib import Path

import numpy as np
from IPython.display import display
from sklearn.linear_model import (
    LinearRegression,
    LogisticRegression,
    SGDClassifier,
    SGDRegressor,
)

from mlektic import (
    explain_logistic_prediction,
    explain_lr_prediction,
    export_figure,
    fit_history,
    fit_history_logistic,
    visualize_logistic,
    visualize_lr,
)

print(f"Mlektic version: {version('mlektic')}")
print("Phase 0 review environment loaded successfully.")

In [ ]:
def summarize_history(history):
    metadata = history["metadata"]
    return {
        "source": metadata["source"],
        "requested_mode": metadata["requested_mode"],
        "resolved_mode": metadata["resolved_mode"],
        "captured_K": metadata["captured_steps"],
        "displayed_N": metadata["displayed_steps"],
        "displayed_coordinates": metadata["displayed_step_indices"].tolist(),
        "training_total_steps": metadata["training_total_steps"],
        "final_state_matches_estimator": metadata["final_state_matches_estimator"],
        "smoothing": metadata["smoothing"],
        "warnings": [item["code"] for item in metadata["warnings"]],
    }

def assert_time_alignment(history):
    n = history["metadata"]["displayed_steps"]
    assert len(history["loss_raw"]) == n
    assert len(history["loss_display"]) == n
    assert len(history["step_indices"]) == n
    assert history["metadata"]["displayed_step_indices"].tolist() == history["step_indices"].tolist()
    return True

RNG = np.random.default_rng(7)

## 1. Linear regression — reconstructed replay

`SGDRegressor` supports `partial_fit`, so `mode="auto"` resolves to an incremental replay over a clone. The replay is not the original `fit` trajectory. We intentionally construct K = 20 checkpoints and display only N = 6.

In [ ]:
X_linear = np.linspace(-2.0, 2.0, 60).reshape(-1, 1)
y_linear = 1.5 + 2.25 * X_linear[:, 0] + RNG.normal(0.0, 0.08, len(X_linear))
sgd_regressor = SGDRegressor(max_iter=30, random_state=7).fit(X_linear, y_linear)

linear_replay = fit_history(
    sgd_regressor,
    X_linear,
    y_linear,
    steps=20,
    max_frames=6,
    smooth="ema",
    smooth_beta=0.80,
)

display(summarize_history(linear_replay))
assert linear_replay["metadata"]["source"] == "replayed"
assert linear_replay["metadata"]["captured_steps"] == 20
assert linear_replay["metadata"]["displayed_steps"] == 6
assert linear_replay["step_indices"][0] == 1
assert linear_replay["step_indices"][-1] == 20
assert linear_replay["metadata"]["source_detail"]["effective_replay_parameters"]["max_iter"] == 1
assert assert_time_alignment(linear_replay)
print("Replay contract: PASS")

In [ ]:
assert linear_replay["loss_raw"] is not linear_replay["loss_display"]
assert not np.allclose(linear_replay["loss_raw"], linear_replay["loss_display"])
np.testing.assert_allclose(linear_replay["loss_hist"], linear_replay["loss_display"])
np.testing.assert_allclose(linear_replay["metrics_hist"]["Loss"], linear_replay["loss_display"])

display({
    "loss_raw": np.round(linear_replay["loss_raw"], 5).tolist(),
    "loss_display": np.round(linear_replay["loss_display"], 5).tolist(),
})
print("Raw/display loss separation: PASS")

In [ ]:
linear_replay_figure = visualize_lr(
    sgd_regressor,
    X_linear,
    y_linear,
    steps=20,
    max_frames=6,
    smooth="ema",
    animation_mode="native",
    show_history_context=True,  # Set to False to hide the subtitle only.
)

assert "Reconstructed replay" in linear_replay_figure.layout.title.text
assert "Reconstructed replay (6/20)" in linear_replay_figure.layout.sliders[0].currentvalue.prefix
display(linear_replay_figure)

## 2. Linear regression — synthetic interpolation

`LinearRegression` does not provide `partial_fit`. Auto mode therefore constructs a baseline-to-fitted-model interpolation. Intermediate states are synthetic, and the final alpha state should match the fitted estimator.

In [ ]:
X_plane = RNG.normal(size=(80, 2))
y_plane = 1.0 + 2.0 * X_plane[:, 0] - 0.8 * X_plane[:, 1]
linear_model = LinearRegression().fit(X_plane, y_plane)

linear_interpolation = fit_history(
    linear_model, X_plane, y_plane, steps=11, max_frames=5, baseline="mean"
)
display(summarize_history(linear_interpolation))

assert linear_interpolation["metadata"]["source"] == "interpolated"
assert linear_interpolation["metadata"]["source_detail"]["baseline"] == "mean"
assert linear_interpolation["metadata"]["final_state_matches_estimator"] is True
np.testing.assert_allclose(linear_interpolation["alpha_values"][[0, -1]], [0.0, 1.0])
assert assert_time_alignment(linear_interpolation)

linear_interpolation_figure = visualize_lr(
    linear_model, X_plane, y_plane, steps=11, max_frames=5
)
assert "Synthetic interpolation" in linear_interpolation_figure.layout.title.text
assert [step.label for step in linear_interpolation_figure.layout.sliders[0].steps] == ["0%", "20%", "50%", "70%", "100%"]
display(linear_interpolation_figure)

## 3. Binary logistic regression — string labels

Scikit-learn permits non-numeric class labels. The history metrics must use `classes_[1]` as the fitted positive class instead of assuming that the positive label is the integer `1`.

In [ ]:
binary_labels = np.where(X_plane[:, 0] + 0.5 * X_plane[:, 1] > 0, "accepted", "rejected")
binary_model = LogisticRegression().fit(X_plane, binary_labels)

binary_history = fit_history_logistic(binary_model, X_plane, binary_labels, steps=9, max_frames=5)
display(summarize_history(binary_history))
assert binary_history["classes"].tolist() == binary_model.classes_.tolist()
assert "F1 Score" in binary_history["metrics_hist"]
assert np.all(np.isfinite(binary_history["metrics_hist"]["F1 Score"]))

binary_figure = visualize_logistic(binary_model, X_plane, binary_labels, steps=9, max_frames=5)
assert binary_figure.layout.meta["mlektic_history"]["source"] == "interpolated"
display(binary_figure)

## 4. Multiclass logistic regression — 2D probability surfaces

This case checks class ordering, multiclass-link metadata, a dense mathematical header, and the repeated source/N/K information in the slider.

In [ ]:
class_scores = np.column_stack([
    X_plane[:, 0],
    X_plane[:, 1],
    -X_plane[:, 0] - X_plane[:, 1],
])
multiclass_labels = np.array(["red", "green", "blue"])[np.argmax(class_scores, axis=1)]
multiclass_model = LogisticRegression().fit(X_plane, multiclass_labels)

multiclass_history = fit_history_logistic(
    multiclass_model, X_plane, multiclass_labels, steps=9, max_frames=5
)
display({
    **summarize_history(multiclass_history),
    "classes": multiclass_history["classes"].tolist(),
    "probability_link": multiclass_history["probability_link"],
})
assert multiclass_history["classes"].tolist() == multiclass_model.classes_.tolist()
assert multiclass_history["probability_link"] in {"softmax", "ovr"}

multiclass_figure = visualize_logistic(
    multiclass_model, X_plane, multiclass_labels, steps=9, max_frames=5
)
assert "Synthetic interpolation (5/9)" in multiclass_figure.layout.sliders[0].currentvalue.prefix
display(multiclass_figure)

## 5. Prediction integrity and extrapolation

A supplied display value must match the estimator by default. An intentional counterfactual requires `prediction_source="provided"`. The figure also states whether the query is outside an observed per-feature training range.

In [ ]:
in_range_explanation = explain_lr_prediction(
    linear_model, X_plane, y_plane, x_query=[[0.25, -0.25]]
)
assert "model-verified" in in_range_explanation.layout.title.text
assert not in_range_explanation.layout.meta["mlektic_prediction"]["outside_training_feature_indices"]
display(in_range_explanation)

outside_query = [[5.0, -5.0]]
extrapolation_explanation = explain_lr_prediction(
    linear_model, X_plane, y_plane, x_query=outside_query
)
assert "Extrapolation" in extrapolation_explanation.layout.title.text
assert extrapolation_explanation.layout.meta["mlektic_prediction"]["outside_training_feature_indices"]
display(extrapolation_explanation)

In [ ]:
try:
    explain_lr_prediction(
        linear_model, X_plane, y_plane, x_query=[[0.0, 0.0]], yhat=-999.0
    )
except ValueError as error:
    assert "does not match" in str(error)
    print(f"Expected verification failure: {error}")
else:
    raise AssertionError("An inconsistent yhat was accepted unexpectedly.")

counterfactual_explanation = explain_lr_prediction(
    linear_model,
    X_plane,
    y_plane,
    x_query=[[0.0, 0.0]],
    yhat=-999.0,
    prediction_source="provided",
)
assert counterfactual_explanation.layout.meta["mlektic_prediction"]["source"] == "provided"
assert "counterfactual" in counterfactual_explanation.layout.title.text
display(counterfactual_explanation)

In [ ]:
logistic_query = [[0.4, 0.1]]
logistic_explanation = explain_logistic_prediction(
    binary_model, X_plane, binary_labels, x_query=logistic_query, show_class_labels=False
)
assert logistic_explanation.layout.meta["mlektic_prediction"]["source"] == "model"
assert logistic_explanation.layout.meta["mlektic_prediction"]["displayed_class"] in binary_model.classes_
assert logistic_explanation.layout.meta["mlektic_prediction"]["show_class_labels"] is False
display(logistic_explanation)

labeled_logistic_explanation = explain_logistic_prediction(
    binary_model, X_plane, binary_labels, x_query=logistic_query, show_class_labels=True
)
assert labeled_logistic_explanation.layout.meta["mlektic_prediction"]["show_class_labels"] is True
display(labeled_logistic_explanation)

## 6. HTML export contract

The default export embeds Plotly but loads MathJax from a CDN. It is therefore not a fully offline mathematical document. Change `RUN_EXPORT` to `True` to write the review artifact.

In [ ]:
RUN_EXPORT = False
EXPORT_PATH = Path("artifacts/phase_0_review.html")

if RUN_EXPORT:
    written_path = export_figure(
        linear_replay_figure,
        EXPORT_PATH,
        include_plotly="inline",
        include_mathjax="cdn",
        responsive=False,
        auto_play=False,
    )
    html = written_path.read_text(encoding="utf-8")
    assert "plotly.js" in html
    assert "MathJax.js" in html
    print(f"Export contract: PASS — {written_path}")
else:
    print("Export skipped. Set RUN_EXPORT=True to create artifacts/phase_0_review.html.")

## 7. Final review summary

The following cell is the compact acceptance gate for this notebook. If it passes, the core phase 0 contracts exercised here are internally consistent. Visual judgment is still required for title spacing, mathematical density, slider readability, and motion in your own notebook environment.

In [ ]:
review_results = {
    "linear_replay_provenance": linear_replay["metadata"]["source"] == "replayed",
    "linear_replay_K_gt_N": linear_replay["metadata"]["captured_steps"] > linear_replay["metadata"]["displayed_steps"],
    "raw_loss_preserved": linear_replay["loss_raw"] is not linear_replay["loss_display"],
    "interpolation_final_matches": linear_interpolation["metadata"]["final_state_matches_estimator"] is True,
    "interpolation_alpha_complete": np.isclose(linear_interpolation["alpha_values"][-1], 1.0),
    "string_label_f1": np.all(np.isfinite(binary_history["metrics_hist"]["F1 Score"])),
    "multiclass_link_declared": multiclass_history["probability_link"] in {"softmax", "ovr"},
    "extrapolation_declared": bool(extrapolation_explanation.layout.meta["mlektic_prediction"]["outside_training_feature_indices"]),
    "counterfactual_declared": counterfactual_explanation.layout.meta["mlektic_prediction"]["source"] == "provided",
}

display(review_results)
assert all(review_results.values()), review_results
print("PHASE 0 NOTEBOOK REVIEW: PASS")

## Suggested manual checklist

After running all cells, inspect the following manually:

1. Replay and interpolation subtitles are readable without covering formulas.
2. Slider labels show source checkpoints or percentages, not renumbered display positions.
3. Animation remains fluid and Play/Pause controls remain stable.
4. Loss and metric values advance in synchronization.
5. The multiclass 2D header remains legible at your notebook width.
6. Extrapolation and counterfactual subtitles are immediately understandable.
7. Classic colors, dimensions, and geometry look unchanged apart from the new transparency labels.

The larger exploratory notebooks remain useful as a second compatibility pass. This notebook is the focused phase 0 acceptance review.